**[🏠 Course Home](00_START_HERE.ipynb)** | [⬅️ Previous: Sheet 4 (Production MCMC)](04_mcmc_production_diagnostics.ipynb) | **Bonus Chapter: Real-World Case Study**

---

# Bonus Chapter: Real-World Bayesian Reliability — Modeling CI Test Flakiness

In **Sheets 1 through 4**, we learned the mathematical machinery of Bayesian data analysis (Grid Approximation, Quadratic Approximation, and MCMC).

In this bonus chapter, we apply Bayesian inference to a critical software engineering problem: **tracking and quarantining flaky tests in Continuous Integration (CI) pipelines** using the exact **Beta-Binomial conjugate model** in real time.

---

## Table of Contents
1. **Part 1: The Problem of Test Flakiness in Software Engineering**
2. **Part 2: The Beta-Binomial Conjugate Model**
3. **Part 3: Interactive Streaming Updates Across CI Runs**
4. **Part 4: Automated CI Quarantine Decision Rules (Tail Risk)**
5. **Part 5: Time-Varying Flakiness with Exponential Memory Decay**
6. **Part 6: Hands-On Challenge Exercises**

## Part 1: The Problem of Test Flakiness

A **flaky test** is an automated test that produces both passes and failures on identical, unmodified code (due to race conditions, network latency, asynchronous timeouts, or environment state).

### Why Naive Heuristics Fail:
- Heuristic: *"Quarantine if it failed twice this week."* $\to$ Punishes heavily run tests and ignores rare flakes on seldom-run tests.
- **The Bayesian Solution**: Treat the test's unknown flake rate $\theta \in [0, 1]$ as a continuous probability distribution and update it dynamically on every single commit.

## Part 2: The Beta-Binomial Conjugate Model

1. **Parameter of Interest**: $\theta \in [0, 1]$ = True flakiness probability.
2. **Likelihood**: For $N$ runs with $k$ failures, outcomes follow a **Binomial distribution**:
   $$P(k \mid N, \theta) = \binom{N}{k} \theta^k (1 - \theta)^{N - k}$$
3. **Prior**: The **Beta distribution** $\text{Beta}(\alpha_0, \beta_0)$:
   - $\alpha_0$ = Prior "pseudo-failures" (e.g. 1).
   - $\beta_0$ = Prior "pseudo-passes" (e.g. 99 for a healthy 1% baseline flakiness rate).
4. **The Algebraic Conjugate Miracle**:
   $$\text{Posterior} \sim \text{Beta}\big(\alpha_0 + \text{Failures}, \;\; \beta_0 + \text{Passes}\big)$$
   Updating beliefs requires **zero calculus, zero grids, and zero MCMC**—just simple integer addition!

In [ ]:
# Define Initial Prior: Beta(1, 99) -> 1% expected flakiness
alpha_0 <- 1
beta_0  <- 99

cat(sprintf("Day 0 Prior Mean Flakiness: %.2f%% [95%% CI: %.2f%% - %.2f%%]\n",
            100 * (alpha_0 / (alpha_0 + beta_0)),
            100 * qbeta(0.025, alpha_0, beta_0),
            100 * qbeta(0.975, alpha_0, beta_0)))

# Visualize 3 Classic Prior Stances for Software Engineering
curve(dbeta(x, 1, 99), from = 0, to = 0.20, col = "darkblue", lwd = 3, las = 1,
      main = "Priors on Test Flakiness (theta)", xlab = "Flake Rate (theta)", ylab = "Density")
curve(dbeta(x, 2, 98), add = TRUE, col = "darkgreen", lwd = 2, lty = 2)
curve(dbeta(x, 1, 1),  add = TRUE, col = "darkred", lwd = 2, lty = 3)

legend("topright", legend = c("Optimistic Suite Prior: Beta(1, 99)", 
                             "Historical Baseline: Beta(2, 98)", 
                             "Uninformative / Flat: Beta(1, 1)"),
       col = c("darkblue", "darkgreen", "darkred"), lty = c(1, 2, 3), lwd = c(3, 2, 2), bty = "n")

## Part 3: Interactive Streaming Updates Across CI Runs

Let us simulate a live CI pipeline tracking a newly written test over time:
- **Week 1 (Solid Behavior)**: 100 test runs, 0 failures $\to$ Flakiness estimate drops towards 0%.
- **Week 2 (Flaky Bug Introduced)**: 25 test runs, 4 intermittent failures $\to$ Flakiness spikes!

In [ ]:
# Start with prior Beta(1, 99)
alpha <- 1
beta  <- 99

# --- Batch 1: 100 Consecutive Passes ---
n1 <- 100; k1 <- 0
alpha_w1 <- alpha + k1
beta_w1  <- beta + (n1 - k1)

# --- Batch 2: 25 Runs with 4 Failures (Flaky Bug) ---
n2 <- 25; k2 <- 4
alpha_w2 <- alpha_w1 + k2
beta_w2  <- beta_w1 + (n2 - k2)

cat("=== Tracking Posterior Evolution ===\n")
cat(sprintf("Day 0 Prior:      Flakiness = %.2f%%\n", 100 * alpha / (alpha + beta)))
cat(sprintf("Week 1 (100 Pass): Flakiness = %.2f%% [95%% CI: %.2f%% - %.2f%%]\n", 
            100 * alpha_w1 / (alpha_w1 + beta_w1),
            100 * qbeta(0.025, alpha_w1, beta_w1),
            100 * qbeta(0.975, alpha_w1, beta_w1)))
cat(sprintf("Week 2 (4 Fails):  Flakiness = %.2f%% [95%% CI: %.2f%% - %.2f%%]\n", 
            100 * alpha_w2 / (alpha_w2 + beta_w2),
            100 * qbeta(0.025, alpha_w2, beta_w2),
            100 * qbeta(0.975, alpha_w2, beta_w2)))

# Visualizing Belief Updates
curve(dbeta(x, alpha, beta), from = 0, to = 0.15, col = "gray60", lty = 2, lwd = 2, las = 1,
      main = "Live Bayesian Updating in CI Pipeline", xlab = "Flakiness Rate (theta)", ylab = "Posterior Density")
curve(dbeta(x, alpha_w1, beta_w1), add = TRUE, col = "darkgreen", lwd = 3)
curve(dbeta(x, alpha_w2, beta_w2), add = TRUE, col = "darkred", lwd = 3)

legend("topright", legend = c("Day 0 Prior: Beta(1, 99)", 
                             "Week 1 (100 Passes): Beta(1, 199)", 
                             "Week 2 (4 Failures): Beta(5, 220)"),
       col = c("gray60", "darkgreen", "darkred"), lty = c(2, 1, 1), lwd = c(2, 3, 3), bty = "n")

## Part 4: Automated CI Quarantine Decision Rules (Tail Risk)

Rather than waiting for manual complaints, CI bots can query the posterior tail probability:
> *"What is the probability that this test's true flakiness rate $\theta$ exceeds $5\%$?"*
$$P(\theta > 0.05 \mid \text{Data}) = 1 - \text{pbeta}(0.05, \alpha, \beta)$$

In [ ]:
# Compute Tail Risk Probability (P(theta > 0.05))
prob_flaky_w1 <- 1 - pbeta(0.05, shape1 = alpha_w1, shape2 = beta_w1)
prob_flaky_w2 <- 1 - pbeta(0.05, shape1 = alpha_w2, shape2 = beta_w2)

cat(sprintf("Week 1: P(Flakiness > 5%%) = %.4f%%\n", 100 * prob_flaky_w1))
cat(sprintf("Week 2: P(Flakiness > 5%%) = %.2f%%\n", 100 * prob_flaky_w2))

# Automated Bot Decision Logic
if (prob_flaky_w2 > 0.90) {
  cat("\n🚨 [CI BOT ACTION]: QUARANTINE TEST! >90% probability that flakiness exceeds 5% SLA threshold.\n")
}

## Part 5: Time-Varying Flakiness with Exponential Memory Decay

Software evolves. If an engineer fixes the underlying race condition on Day 10, past failures shouldn't permanently taint the test's reputation.

We introduce a **daily memory decay factor $\gamma \in [0.90, 0.98]$**:
$$\alpha_{t+1} = \gamma \cdot \alpha_t + \text{Failures}_{\text{today}}$$
$$\beta_{t+1} = \gamma \cdot \beta_t + \text{Passes}_{\text{today}}$$

In [ ]:
# Simulate 14 Days with a Fixed Bug on Day 7
gamma <- 0.92  # 8% memory decay per day

alpha_curr <- alpha_w2
beta_curr  <- beta_w2

history_mean <- numeric(14)
history_ci_high <- numeric(14)

# Days 1 to 14: Bug fixed, all 20 runs per day PASS
for (day in 1:14) {
  # Decay old pseudo-counts and add 20 passes
  alpha_curr <- gamma * alpha_curr + 0
  beta_curr  <- gamma * beta_curr  + 20
  
  history_mean[day]    <- 100 * (alpha_curr / (alpha_curr + beta_curr))
  history_ci_high[day] <- 100 * qbeta(0.975, alpha_curr, beta_curr)
}

# Plotting Recovery Trajectory after Bug Fix
plot(1:14, history_mean, type = "b", col = "darkblue", pch = 19, lwd = 3, las = 1, ylim = c(0, 8),
     main = "Flakiness Decay & Recovery After Bug Fix (gamma = 0.92)",
     xlab = "Days Since Fix (20 Passes/Day)", ylab = "Estimated Flakiness (%)")
lines(1:14, history_ci_high, col = "darkred", lty = 2, lwd = 2)
abline(h = 1.0, col = "darkgreen", lty = 3, lwd = 2)

legend("topright", legend = c("Expected Flakiness Mean", "97.5% Upper Credible Bound", "1% Baseline Healthy Target"),
       col = c("darkblue", "darkred", "darkgreen"), lty = c(1, 2, 3), pch = c(19, NA, NA), lwd = c(3, 2, 2), bty = "n")

cat(sprintf("Day 1 Flakiness: %.2f%% -> Day 14 Flakiness: %.2f%% (Successfully De-quarantined!)\n", 
            history_mean[1], history_mean[14]))

## Part 6: Hands-On Challenge Exercises

### Exercise 1: Finding De-Quarantine Day
Using the decay code above, find the exact day when the upper 95% Credible Interval bound falls below **$2.0\%$** so the CI bot can automatically un-quarantine the test.

### Exercise 2: Hierarchical Flakiness Across 1,000 Tests
In a suite of 1,000 tests, why is a **Hierarchical Bayesian model** (using MCMC in Stan) better than fitting 1,000 independent Beta priors? *(Hint: Partial pooling shares information across the entire repository to learn the suite's global quality distribution).* 

---

**[🏠 Course Home](00_START_HERE.ipynb)** | [⬅️ Previous: Sheet 4 (Production MCMC)](04_mcmc_production_diagnostics.ipynb) | **Masterclass Complete 🎉**